# Telemetry Data Analysis

This notebook analyzes player telemetry data to calculate play duration and identify players with short sessions.

**Assumption**: Each telemetry row represents a 30-second window.

In [152]:
%pip install pandas
import pandas as pd
import os

Note: you may need to restart the kernel to use updated packages.


In [153]:
# Configuration
TELEMETRY_PATH = 'data/telemetry_phase_2.telemetries.csv'
USERS_PATH = 'data/telemetry_phase_2.users.csv'
OUTPUT_PATH = 'data/processed/processed_telemetry_data.csv'

X_MINUTES = 30 # Minimum Threshold: Filter players <= X
Y_MINUTES = 45 # Maximum Cap: Filter players > Y (Trim unless rich)

# Columns to check for "Richness"
RICH_COLUMNS = ['itemsCollected', 'kills', 'rawJson.deaths']

In [154]:
# Load Data
if not os.path.exists(TELEMETRY_PATH) or not os.path.exists(USERS_PATH):
    print(f"Error: Data files not found.")
    print(f"Looking for: {TELEMETRY_PATH}")
    print(f"Looking for: {USERS_PATH}")
else:
    df_telemetry = pd.read_csv(TELEMETRY_PATH)
    df_users = pd.read_csv(USERS_PATH)
    print(f"Successfully loaded {len(df_telemetry)} telemetry rows and {len(df_users)} users.")

Successfully loaded 2077 telemetry rows and 37 users.


In [155]:
# Process Data
if 'userId' in df_telemetry.columns and '_id' in df_users.columns:
    # 1. Calculate Duration per User (Count * 30s)
    user_activity = df_telemetry.groupby('userId').size().reset_index(name='data_points')
    user_activity['duration_seconds'] = user_activity['data_points'] * 30
    user_activity['duration_minutes'] = user_activity['duration_seconds'] / 60

    # 2. Merge with User Details
    merged_data = pd.merge(
        user_activity,
        df_users[['_id', 'firstName', 'lastName']],
        left_on='userId',
        right_on='_id',
        how='inner'
    )
    
    print("Data processed and merged successfully.")
    display(merged_data.head())
else:
    print("Error: Missing required columns ('userId' in telemetry, '_id' in users).")

Data processed and merged successfully.


,userId,data_points,duration_seconds,duration_minutes,_id,firstName,lastName
0,6974892348d53c4152cf1421,105,3150,52.5,6974892348d53c4152cf1421,Sithmi,Kashmira
1,6974abcf315a39e91bc1e471,9,270,4.5,6974abcf315a39e91bc1e471,Nethu,Samarathunge
2,6974acd8315a39e91bc1e52f,157,4710,78.5,6974acd8315a39e91bc1e52f,Sakuna,Yasasvi
3,6974f3cc5d8e98b89f0a5d07,103,3090,51.5,6974f3cc5d8e98b89f0a5d07,Vidusara,Wijerathnayake
4,6974f5555d8e98b89f0a5fa5,58,1740,29.0,6974f5555d8e98b89f0a5fa5,Kavinda,Ravihansa


In [156]:
# Filter and Display Results (Analysis View)
if 'duration_minutes' in merged_data.columns:
    short_playtime = merged_data[merged_data['duration_minutes'] <= X_MINUTES].copy()
    target_playtime = merged_data[(merged_data['duration_minutes'] > X_MINUTES) & (merged_data['duration_minutes'] <= Y_MINUTES)].copy()
    capped_playtime = merged_data[merged_data['duration_minutes'] > Y_MINUTES].copy()
    
    # Sort all lists
    short_playtime.sort_values(by='duration_minutes', ascending=False, inplace=True)
    target_playtime.sort_values(by='duration_minutes', ascending=False, inplace=True)
    capped_playtime.sort_values(by='duration_minutes', ascending=False, inplace=True)

    print(f"\n--- Analysis Report ---")
    print(f"Short (<= {X_MINUTES}m): {len(short_playtime)}")
    print(f"Target (> {X_MINUTES}m & <= {Y_MINUTES}m): {len(target_playtime)}")
    print(f"Long (> {Y_MINUTES}m): {len(capped_playtime)}")
    
    print(f"\n--- [Target Playtime] ({X_MINUTES} < d <= {Y_MINUTES} mins) ---")
    display(target_playtime[['firstName', 'lastName', 'duration_minutes', 'data_points']])

    print(f"\n--- [Long Playtime] (> {Y_MINUTES} mins) ---")
    display(capped_playtime[['firstName', 'lastName', 'duration_minutes', 'data_points']])
else:
    print("Skipping analysis due to missing data.")


--- Analysis Report ---
Short (<= 30m): 18
Target (> 30m & <= 45m): 12
Long (> 45m): 6

--- [Target Playtime] (30 < d <= 45 mins) ---


,firstName,lastName,duration_minutes,data_points
25,Nisandu,Senanayake,42.0,84
15,Supun,Gamage,41.0,82
29,Dulina,Dias,41.0,82
17,Ranura,Indudunu,38.5,77
9,Kaveesh,Jayawardana,37.5,75
23,Thimanya,Senanayake,36.0,72
26,Levidu,Wickramasinghe,35.5,71
24,Azka,Afham,35.5,71
7,Yohan,Wanasinghe,34.5,69
22,Samadhi,Anuranga,33.0,66



--- [Long Playtime] (> 45 mins) ---


,firstName,lastName,duration_minutes,data_points
2,Sakuna,Yasasvi,78.5,157
12,Nimeth,Dasanayake,66.0,132
6,Pasindu,Godakanda,57.0,114
0,Sithmi,Kashmira,52.5,105
3,Vidusara,Wijerathnayake,51.5,103
18,Shehan,Sanjula,49.0,98


In [157]:
# Data Processing & Export Logic
# 1. Filter Users: Keep ONLY users > X minutes
valid_users = merged_data[merged_data['duration_minutes'] > X_MINUTES]['userId'].unique()
final_rows = []

print(f"\n--- Processing Data for Export ---")
print(f"Processing {len(valid_users)} users (excluding Short Playtime)...")

Y_POINTS = int((Y_MINUTES * 60) / 30) # Max points (e.g., 45 mins * 2 = 90 points)

for uid in valid_users:
    # Get user's telemetry rows, sorted by timestamp (if available) or assume index order
    user_rows = df_telemetry[df_telemetry['userId'] == uid].copy()
    
    # If timestamp exists, sort. If not, rely on CSV order.
    if 'timestamp' in user_rows.columns:
        user_rows.sort_values(by='timestamp', inplace=True)

    total_count = len(user_rows)
    
    # 2. Check Cap Condition
    if total_count <= Y_POINTS:
        # User is within Target Range (or slightly over X but under Y). Keep ALL.
        final_rows.append(user_rows)
    else:
        # User Exceeded Y. Check for "Richness" in the excess tail.
        # Tail = data beyond Y_POINTS
        tail_rows = user_rows.iloc[Y_POINTS:]
        
        # Check if tail has significant events (Sum of rich cols > 0)
        is_tail_rich = False
        for col in RICH_COLUMNS:
            if col in tail_rows.columns:
                # If any value check is non-zero, assume rich
                if tail_rows[col].sum() > 0:
                    is_tail_rich = True
                    break
        
        if is_tail_rich:
            # RICH DATA: Keep ALL rows (Don't trim)
            final_rows.append(user_rows)
            print(f"  [Rich Data] Kept full history for user {uid} ({total_count} pts)")
        else:
            # IDLE/EMPTY TAIL: Trim to Y_POINTS
            trimmed_rows = user_rows.iloc[:Y_POINTS]
            final_rows.append(trimmed_rows)
            print(f"  [Trimmed] User {uid}: {total_count} -> {Y_POINTS} pts")

# 3. Concatenate and Save
if final_rows:
    df_final = pd.concat(final_rows)
    df_final.to_csv(OUTPUT_PATH, index=False)
    print(f"\nProcessing Complete.")
    print(f"Total Data Points Exported: {len(df_final)}")
    print(f"Saved to: {OUTPUT_PATH}")
else:
    print("No data to export.")


--- Processing Data for Export ---
Processing 18 users (excluding Short Playtime)...
  [Rich Data] Kept full history for user 6974892348d53c4152cf1421 (105 pts)
  [Rich Data] Kept full history for user 6974acd8315a39e91bc1e52f (157 pts)
  [Rich Data] Kept full history for user 6974f3cc5d8e98b89f0a5d07 (103 pts)
  [Rich Data] Kept full history for user 697502455d8e98b89f0a9f38 (114 pts)
  [Rich Data] Kept full history for user 69758a65547a2b541d10f4ed (132 pts)
  [Trimmed] User 69759b30547a2b541d1151be: 98 -> 90 pts

Processing Complete.
Total Data Points Exported: 1575
Saved to: data/processed/processed_telemetry_data.csv
